# FitCoach — Push Models to HuggingFace Hub

Pushes both FitCoach models to HF Hub so they can be used in the Gradio Space.

| Model | What we push | HF Repo |
|-------|-------------|---------|
| 3B | Merged 16-bit weights | `Harsh-k-007/fitcoach-3b` |
| 8B | LoRA adapter only | `Harsh-k-007/fitcoach-8b-adapter` |

**Expected time:** ~30–45 min total (mostly upload time, depends on internet speed)  
**No GPU needed** — this is just file upload, CPU runtime is fine

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install huggingface_hub transformers

## Step 2 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Paths
PATH_3B = '/content/drive/MyDrive/fitcoach_data/fitcoach-llama3.2-3b/merged_16bit'
PATH_8B = '/content/drive/MyDrive/fitcoach_data/fitcoach-llama3.1-8b/lora_adapter'

# HF repo names
REPO_3B = 'Harsh-k-007/fitcoach-3b'
REPO_8B = 'Harsh-k-007/fitcoach-8b-adapter'

# Sanity checks
assert os.path.exists(os.path.join(PATH_3B, 'model-00001-of-00002.safetensors')), \
    '3B shard 1 not found — check PATH_3B'
assert os.path.exists(os.path.join(PATH_3B, 'model-00002-of-00002.safetensors')), \
    '3B shard 2 not found — check PATH_3B'
assert os.path.exists(os.path.join(PATH_8B, 'adapter_model.safetensors')), \
    '8B adapter not found — check PATH_8B'

print('✅ All model files found')
print(f'   3B path : {PATH_3B}')
print(f'   8B path : {PATH_8B}')

## Step 3 — HuggingFace login

Use a token with **write** permission (needed to create repos and upload files).  
Get one at: https://huggingface.co/settings/tokens → New token → Role: Write

In [ ]:
from huggingface_hub import login, HfApi

login()  # paste your HF token with WRITE permission

api = HfApi()
print('✅ Logged in')

## Step 4 — Create HF repos

Creates the two repos on HF Hub if they don't exist yet. Safe to re-run.

In [ ]:
from huggingface_hub import create_repo

for repo_id in [REPO_3B, REPO_8B]:
    try:
        create_repo(repo_id, repo_type='model', exist_ok=True)
        print(f'✅ Repo ready: https://huggingface.co/{repo_id}')
    except Exception as e:
        print(f'❌ Error creating {repo_id}: {e}')

## Step 5 — Push 3B merged model

Uploads ~6GB total (two shards + configs). Takes ~15–25 min depending on Colab's upload speed.  
You'll see a progress bar per file.

In [ ]:
from huggingface_hub import upload_folder

print(f'Uploading 3B model to {REPO_3B}...')
upload_folder(
    folder_path=PATH_3B,
    repo_id=REPO_3B,
    repo_type='model',
    commit_message='Upload FitCoach 3B merged 16-bit model',
    ignore_patterns=['.cache/*'],  # skip the cache folder
)
print(f'\n✅ 3B model uploaded!')
print(f'   View at: https://huggingface.co/{REPO_3B}')

## Step 6 — Push 8B LoRA adapter

Uploads ~177MB total (adapter weights + configs). Takes ~2–5 min.

In [ ]:
print(f'Uploading 8B adapter to {REPO_8B}...')
upload_folder(
    folder_path=PATH_8B,
    repo_id=REPO_8B,
    repo_type='model',
    commit_message='Upload FitCoach 8B LoRA adapter',
)
print(f'\n✅ 8B adapter uploaded!')
print(f'   View at: https://huggingface.co/{REPO_8B}')

## Step 7 — Verify both repos

Lists files in each repo to confirm the upload completed correctly.

In [ ]:
from huggingface_hub import list_repo_files

for repo_id, label in [(REPO_3B, '3B merged'), (REPO_8B, '8B adapter')]:
    print(f'\n--- {label} ({repo_id}) ---')
    files = list(list_repo_files(repo_id))
    for f in sorted(files):
        print(f'  {f}')

print('\n✅ Both repos verified — ready to build the Gradio app!')